# Neural Network

A documented negative result: this notebook tries a class-weighted neural network, then focal loss, then a deeper architecture, none of which beat RF or XGBoost. Kept here (cleaned up) rather than deleted, because understanding *why* it lost is a real finding, not just noise.

## 1. Load Data

In [ ]:
import sys
sys.path.append('../src')
from data_prep import load_data

X_train, X_test, y_train, y_test, amount_test = load_data()

## 2. Establish Baseline

A small dense network with class weighting to counter the imbalance, instead of resampling.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# random_state=42 for RF/XGBoost has no NN equivalent unless we seed TF explicitly --
# without this, weight init, dropout, and batch shuffling are all uncontrolled and every run differs
tf.random.set_seed(42)
np.random.seed(42)

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = {i: class_weights[i] for i in range(len(classes))}
print("Class weights:", class_weights)

nn_baseline = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(16, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(8, activation='relu'),

    layers.Dense(1, activation='sigmoid')
])

nn_baseline.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['Precision', 'Recall', 'AUC']
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = nn_baseline.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=2048,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

nn_baseline_probs = nn_baseline.predict(X_test).ravel()
y_pred_baseline = (nn_baseline_probs > 0.5).astype(int)

print("Classification Report (default threshold = 0.5):")
print(classification_report(y_test, y_pred_baseline))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_baseline))

## 3. PR-AUC & Threshold Tuning

Same treatment as RF and XGBoost, for a fair comparison -- F1-optimal threshold, not an arbitrary recall floor.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

pr_auc_baseline = average_precision_score(y_test, nn_baseline_probs)
print(f"PR-AUC (Average Precision): {pr_auc_baseline:.4f}")

precision, recall, thresholds = precision_recall_curve(y_test, nn_baseline_probs)

f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_f1_idx = np.argmax(f1_scores)
best_f1_threshold = thresholds[best_f1_idx]

print(f"\nF1-maximizing threshold: {best_f1_threshold:.4f}")
print(f"Precision: {precision[best_f1_idx]:.4f}  Recall: {recall[best_f1_idx]:.4f}  F1: {f1_scores[best_f1_idx]:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label="PR curve")
plt.scatter(
    recall[best_f1_idx], precision[best_f1_idx],
    color="red", zorder=5, label=f"F1-optimal (t={best_f1_threshold:.3f})"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("NN Baseline: Precision-Recall Curve")
plt.legend()
plt.show()

## 4. Attempt: Focal Loss

Focal loss down-weights easy examples and focuses training on hard ones -- designed for exactly this kind of severe imbalance. Same architecture, retrained with focal loss instead of class-weighted binary cross-entropy.

In [ ]:
from tensorflow.keras import backend as K

tf.random.set_seed(42)

def focal_loss(gamma=2., alpha=.25):
    def focal_loss_fixed(y_true, y_pred):
        y_pred = K.clip(y_pred, K.epsilon(), 1 - K.epsilon())
        cross_entropy = -y_true * K.log(y_pred) - (1 - y_true) * K.log(1 - y_pred)
        weight = alpha * y_true * K.pow((1 - y_pred), gamma) + \
                 (1 - alpha) * (1 - y_true) * K.pow(y_pred, gamma)
        return K.mean(weight * cross_entropy)
    return focal_loss_fixed

nn_focal = models.clone_model(nn_baseline)
nn_focal.compile(optimizer='adam', loss=focal_loss(gamma=2, alpha=0.25), metrics=['accuracy'])

history_focal = nn_focal.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=2048,
    callbacks=[early_stop],
    verbose=1
)

nn_focal_probs = nn_focal.predict(X_test).ravel()
pr_auc_focal = average_precision_score(y_test, nn_focal_probs)
print(f"PR-AUC (Average Precision): {pr_auc_focal:.4f}")

**Finding: focal loss helps, consistently, though the exact numbers vary run to run.** Across multiple runs (even with `tf.random.set_seed(42)`), focal loss PR-AUC has landed around 0.73–0.79 vs. the class-weighted baseline's ~0.69–0.70 — a real, repeatable improvement, just not an exact one. Unlike RF/XGBoost's `random_state=42` (which makes them bit-for-bit reproducible), TensorFlow on CPU uses parallelized floating-point operations (oneDNN) whose summation order isn't fully deterministic even with a fixed seed — so treat any single PR-AUC value here as illustrative, and the *ranking* between attempts as the trustworthy result.

Also worth noting the methodology: this retrains from a fresh random initialization (`clone_model`, not a continued fine-tune of the baseline's already-trained weights), which is a stricter test than just swapping the loss function on an already-converged model. That it consistently helped anyway suggests focal loss's per-example, evolving weighting (focus harder on whatever's still being misclassified) genuinely outperforms a single static per-class weight for this problem.

Still nowhere close to RF or XGBoost, though — see the conclusion below.

## 5. Attempt: Deeper Architecture

More capacity (64→32→16 instead of 32→16→8), same focal loss, to test whether capacity was the limiting factor.

In [ ]:
tf.random.set_seed(42)

nn_deep = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(16, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(1, activation='sigmoid')
])

nn_deep.compile(optimizer='adam', loss=focal_loss(gamma=2, alpha=0.25), metrics=['accuracy'])

history_deep = nn_deep.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=2048,
    callbacks=[early_stop],
    verbose=1
)

nn_deep_probs = nn_deep.predict(X_test).ravel()
pr_auc_deep = average_precision_score(y_test, nn_deep_probs)
print(f"PR-AUC (Average Precision): {pr_auc_deep:.4f}")

## 6. Compare the Three Attempts, Pick the Best

In [ ]:
results = {
    "baseline (class-weighted)": (nn_baseline, nn_baseline_probs, pr_auc_baseline),
    "focal loss": (nn_focal, nn_focal_probs, pr_auc_focal),
    "deeper + focal loss": (nn_deep, nn_deep_probs, pr_auc_deep),
}

for name, (_, _, pr_auc) in results.items():
    print(f"{name}: PR-AUC = {pr_auc:.4f}")

best_name = max(results, key=lambda k: results[k][2])
nn_model, nn_probs, nn_pr_auc = results[best_name]
print(f"\nBest of the three: {best_name} (PR-AUC = {nn_pr_auc:.4f})")

## 7. Cost-Based Threshold

Same cost framework as RF and XGBoost, applied to the best-performing NN variant, for a fair final comparison.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

precision, recall, thresholds = precision_recall_curve(y_test, nn_probs)

REVIEW_COST = 10  # assumed dollar cost to investigate one flagged transaction -- adjust to your own estimate

y_test_arr = y_test.values
amount_arr = amount_test.values

costs = []
for t in thresholds:
    y_pred_t = (nn_probs >= t).astype(int)
    fn_mask = (y_pred_t == 0) & (y_test_arr == 1)
    fp_mask = (y_pred_t == 1) & (y_test_arr == 0)
    cost = amount_arr[fn_mask].sum() + REVIEW_COST * fp_mask.sum()
    costs.append(cost)

costs = np.array(costs)
best_cost_idx = np.argmin(costs)
best_cost_threshold = thresholds[best_cost_idx]

print(f"Cost-minimizing threshold: {best_cost_threshold:.4f}")
print(f"Total cost at this threshold: ${costs[best_cost_idx]:,.2f}")
print(f"Precision: {precision[best_cost_idx]:.4f}  Recall: {recall[best_cost_idx]:.4f}")

y_pred_cost = (nn_probs >= best_cost_threshold).astype(int)
print("\nConfusion Matrix (cost-optimal threshold):")
print(confusion_matrix(y_test, y_pred_cost))
print("\nClassification Report (cost-optimal threshold):")
print(classification_report(y_test, y_pred_cost))

## 8. Conclusion

**None of the three neural network variants beat RF or XGBoost, in any run.** The best of the three (focal loss) has landed a cost-optimal total cost anywhere from ~$2,100 to ~$2,180 across runs (precision ~0.71–0.78, recall ~0.82–0.85) — consistently and clearly worse than RF's $1,969.72 or XGBoost's ~$1,971, both of which get there with far fewer false positives at a similar recall. That gap is large enough to survive the run-to-run noise described above; there's no seed or lucky run that closes it.

The internal story among the three NN attempts is more nuanced than "nothing worked," though, and this part of the finding *is* consistent across every run:
- **Focal loss helped** (PR-AUC roughly 0.69–0.70 → 0.73–0.79) — a real, repeatable gain over the naive class-weighted baseline.
- **More capacity hurt** (PR-AUC dropped further, to roughly 0.60–0.68) — the deeper 64→32→16 network, trained the same way, did worse than the smaller one, in every run. With only ~394 fraud examples in the full training set, a larger network has more room to overfit or struggle to optimize well in a fixed 20 epochs, rather than more room to learn genuine structure.

So capacity wasn't the bottleneck — if anything, less was more. The more likely explanation is the one from the original exploration: PCA-compressed features and a very small, possibly hard-to-cleanly-separate positive class favor tree ensembles, which can directly carve out axis-aligned decision boundaries, over neural networks, which have to learn that structure from scratch with very few positive examples to learn it from.

This isn't a wasted effort — it's evidence that a neural network's extra complexity isn't buying anything here, which is exactly the kind of result worth knowing before deploying something more expensive to train, maintain, and reproduce than a Random Forest.

## 9. Save Model & Predictions

Predictions are what `06_model_comparison.ipynb` actually needs; the Keras model is saved for completeness.

In [ ]:
import json

nn_model.save('../models/nn.keras')
np.save('../predictions/nn_probs.npy', nn_probs)

with open('../models/nn_thresholds.json', 'w') as f:
    json.dump({
        "model_selected": best_name,
        "pr_auc": float(nn_pr_auc),
        "cost_optimal_threshold": float(best_cost_threshold),
        "review_cost_assumption": REVIEW_COST,
    }, f, indent=2)